# Assignment 13: 20M LLM with reversible training

This notebook is the fresh-clone reproduction path for the assignment. It does not assume that the experiment source folder has already been copied to Google Drive. It clones the submitted GitHub repository, keeps checkpoints in Drive, verifies the external packed corpus, tests the reversible implementation, re-measures the maximum batch on the assigned GPU, and prints the final comparison table.

The published local experiment used an RTX 3070 Laptop GPU and produced the reference results stored in `results_summary.json`. A Colab GPU can produce different throughput and maximum-batch values; those values must be measured again rather than copied from the local run.

## Before running

1. Select **Runtime > Change runtime type > GPU**.
2. The submitted repository includes the minimal training-ready dataset through Git LFS. If you intentionally omit it, place `Corpus_20M_v1` and `Run_20M_50M_v1` in Google Drive under `MyDrive/ERA/outputs/` as documented in the README.
3. Run every cell in order. The notebook defaults to this submission repository; set the `ASSIGNMENT13_REPO_URL` environment variable only when testing a fork.

The training cell is intentionally explicit: running it starts multiple 5M/50M-token jobs and can take several GPU-hours. All run state is written to Drive, so the cell can be rerun after a disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, subprocess

if shutil.which('git-lfs') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'git-lfs'], check=True)
subprocess.run(['git', 'lfs', 'install'], check=True)

DEFAULT_REPO_URL = 'https://github.com/udisinghania/ERAv5.git'
REPO_URL = os.environ.get('ASSIGNMENT13_REPO_URL', DEFAULT_REPO_URL).strip()
if not REPO_URL.startswith(('https://github.com/', 'http://github.com/')):
    raise ValueError('ASSIGNMENT13_REPO_URL must be a full GitHub HTTPS clone URL.')
if '/tree/' in REPO_URL or '/blob/' in REPO_URL:
    raise ValueError('Use the repository clone URL, not a GitHub folder or file URL.')

REPO_ROOT = Path('/content/assignment13-repo')
if (REPO_ROOT / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
elif REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
    raise RuntimeError(f'{REPO_ROOT} exists but is not a Git clone; choose a fresh runtime.')
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_ROOT)], check=True)
subprocess.run(['git', '-C', str(REPO_ROOT), 'lfs', 'pull'], check=True)

PROJECT_ROOT = REPO_ROOT / 'Week 13'
if not (PROJECT_ROOT / 'requirements.txt').is_file():
    PROJECT_ROOT = REPO_ROOT
if not (PROJECT_ROOT / 'requirements.txt').is_file():
    raise FileNotFoundError(f'Could not locate the assignment folder under {REPO_ROOT}.')
print('Repository:', REPO_ROOT)
print('Assignment folder:', PROJECT_ROOT)

In [ ]:
import sys, platform, json
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.txt')], check=True)

import torch, numpy as np
assert torch.cuda.is_available(), 'Enable a GPU runtime first.'
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('NumPy:', np.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('Compute capability:', torch.cuda.get_device_capability(0))
print('BF16 supported:', torch.cuda.is_bf16_supported())
if not torch.cuda.is_bf16_supported():
    raise RuntimeError('This definitive experiment requires a CUDA GPU with BF16 support.')

## Create a clean, resumable run directory

The Git clone stays clean in `/content`. Source files are copied into a separate Google Drive directory where checkpoints and logs persist. Change `RUN_LABEL` when you intentionally want a completely new experiment. Reusing the same label resumes interrupted runs.

In [ ]:
DRIVE_OUTPUTS = Path('/content/drive/MyDrive/ERA/outputs')
RUN_LABEL = 'colab_assignment13_v1'
RUN_ROOT = DRIVE_OUTPUTS / f'Reversibility_20M_50M_{RUN_LABEL}'
CORPUS_ROOT = DRIVE_OUTPUTS / 'Corpus_20M_v1'
VALIDATION_ROOT = DRIVE_OUTPUTS / 'Run_20M_50M_v1' / 'validation'
RUN_ROOT.mkdir(parents=True, exist_ok=True)

REPO_CORPUS = PROJECT_ROOT / 'data' / 'Corpus_20M_v1'
REPO_VALIDATION = PROJECT_ROOT / 'data' / 'Run_20M_50M_v1' / 'validation'
if REPO_CORPUS.is_dir():
    shutil.copytree(REPO_CORPUS, CORPUS_ROOT, dirs_exist_ok=True)
if REPO_VALIDATION.is_dir():
    shutil.copytree(REPO_VALIDATION, VALIDATION_ROOT, dirs_exist_ok=True)

source_files = sorted(PROJECT_ROOT.glob('*.py'))
required_source = {'model.py', 'model_precise.py', 'experiment_precise.py', 'run_precise_suite.py', 'test_precise.py'}
present = {p.name for p in source_files}
missing_source = sorted(required_source - present)
if missing_source:
    raise FileNotFoundError(f'Repository is missing source files: {missing_source}')
for src in source_files:
    shutil.copy2(src, RUN_ROOT / src.name)

required_data = [
    CORPUS_ROOT / 'packed_dataset.py',
    CORPUS_ROOT / 'packed_50m_ctx512' / 'packing_report.json',
    CORPUS_ROOT / 'packed_50m_ctx512' / 'input_ids.uint16.bin',
    CORPUS_ROOT / 'packed_50m_ctx512' / 'loss_mask.uint8.bin',
    CORPUS_ROOT / 'packed_50m_ctx512' / 'segment_ids.int16.bin',
    CORPUS_ROOT / 'packed_50m_ctx512' / 'position_ids.uint16.bin',
    CORPUS_ROOT / 'baseline' / 'artifacts' / 'tokenizer_v2' / 'tokenizer.json',
    VALIDATION_ROOT / 'manifest.json',
    VALIDATION_ROOT / 'input_ids.bin',
    VALIDATION_ROOT / 'loss_mask.bin',
    VALIDATION_ROOT / 'segment_ids.bin',
    VALIDATION_ROOT / 'position_ids.bin',
]
missing_data = [str(p) for p in required_data if not p.is_file()]
if missing_data:
    raise FileNotFoundError('Missing external data artifacts:\n' + '\n'.join(missing_data))
os.chdir(RUN_ROOT)
print('Persistent run directory:', RUN_ROOT)
print('Source files copied:', len(source_files))
print('Data contract: PASS')

## Published reference result

This table is read from the small, version-controlled result summary. It lets a fresh clone inspect the submitted measurements without downloading multi-gigabyte checkpoints.

In [ ]:
import pandas as pd
reference = json.loads((PROJECT_ROOT / 'results_summary.json').read_text())
display(pd.DataFrame(reference['controlled_runs']))
print('Selected reversible variant:', reference['selected_reversible_variant'])
print('Verification:', reference['verification'])

## Correctness gate

These tests compare reversible backward against a stored-activation reference, check reconstruction and gradient error, verify causal masking/label shifting, and confirm that saved stack activation bytes do not grow with depth. Run this before spending GPU time.

In [ ]:
subprocess.run([sys.executable, '-B', 'test_precise.py'], cwd=RUN_ROOT, check=True)
correctness = json.loads((RUN_ROOT / 'correctness_precise.json').read_text())
assert correctness['status'] == 'PASS'
print('Correctness gate: PASS')

## Run the complete core assignment

The suite first trains 5M-token midpoint and Euler pilots and selects the lower-loss variant. It then searches for the selected variant's maximum batch on the current GPU, confirms the boundary on packed data, and performs three independent 50M-token runs: baseline batch 32, selected reversible batch 32, and selected reversible maximum batch.

The local extended report additionally trained both reversible variants at their respective maxima. The default Colab path below reproduces exactly what the assignment asks while avoiding unnecessary duplicate multi-hour runs.

In [ ]:
RUN_FULL_EXPERIMENT = True  # Set False if you only want to inspect the submitted reference results.
if RUN_FULL_EXPERIMENT:
    subprocess.run([sys.executable, '-B', 'run_precise_suite.py'], cwd=RUN_ROOT, check=True)
else:
    print('Full training skipped by RUN_FULL_EXPERIMENT=False')

## Verify and report the fresh run

This cell performs assignment-level checks and builds the requested loss, speed, and peak-memory table from the new run. It deliberately discovers the selected variant and measured batch from the suite status instead of assuming the RTX 3070 values.

In [ ]:
if RUN_FULL_EXPERIMENT:
    status = json.loads((RUN_ROOT / 'suite_status_precise.json').read_text())
    assert status['state'] == 'COMPLETE', status
    variant = status['selected_variant']
    max_batch = status.get('max_batch', status.get(f'{variant}_max_batch'))
    run_names = [
        'baseline_precise_b32',
        f'{variant}_precise_b32',
        f'{variant}_precise_b{max_batch}',
    ]
    rows = []
    for name in run_names:
        result = json.loads((RUN_ROOT / name / 'result.json').read_text())
        assert result['state'] == 'COMPLETE'
        assert result['parameters'] == 20_166_912
        assert result['training_tokens'] == 50_000_000
        assert result['final_validation']['tokens'] == 5_049_456
        if result['variant'] != 'baseline':
            audit = json.loads((RUN_ROOT / name / 'gradient_audit.json').read_text())
            assert audit['status'] == 'PASS'
        rows.append({
            'run': name,
            'variant': result['variant'],
            'batch': result['batch'],
            'updates': result['updates'],
            'final_loss': result['final_validation']['loss'],
            'perplexity': result['final_validation']['perplexity'],
            'target_tokens_per_second': result['target_tokens_per_second'],
            'peak_allocated_gib': result['peak_allocated_bytes'] / 2**30,
            'training_minutes': result['training_seconds'] / 60,
        })
    fresh_results = pd.DataFrame(rows)
    display(fresh_results.style.format({
        'final_loss': '{:.4f}',
        'perplexity': '{:.2f}',
        'target_tokens_per_second': '{:,.0f}',
        'peak_allocated_gib': '{:.2f}',
        'training_minutes': '{:.2f}',
    }))
    print('Selected reversible variant:', variant)
    print('Measured maximum batch on this GPU:', max_batch)
    print('Assignment verification: PASS')

## Interpretation

Reversible training saves memory because the custom backward pass reconstructs earlier two-stream states instead of retaining every stack activation from forward. It still stores parameters, gradients, optimizer state, logits, masks, the final reversible state, and active workspace. The trade-off is additional computation during backward. Therefore compare fixed-batch rows to isolate the memory/speed trade-off; treat maximum-batch rows as capacity demonstrations because their update counts and optimization noise differ.